# FinancialPhraseBank EDA & Baseline Evaluation

Use this notebook to inspect the FinancialPhraseBank dataset, run a classical baseline, and evaluate a pretrained FinBERT model. Edit CONFIG if you want a different agreement split (50/66/75/all).

In [ ]:
from datasets import load_dataset
import pandas as pd

CONFIG = "sentences_75agree"  # options: sentences_50agree, sentences_66agree, sentences_75agree, sentences_allagree

ds = load_dataset("takala/financial_phrasebank", CONFIG)
df = ds["train"].to_pandas().rename(columns={"sentence": "text"})
label_map = {0: "negative", 1: "neutral", 2: "positive"}
df["label_text"] = df["label"].map(label_map)
df.head()

In [ ]:
df["label_text"].value_counts()

In [ ]:
from sklearn.model_selection import train_test_split

train_df, test_df = train_test_split(
    df,
    test_size=0.2,
    random_state=42,
    stratify=df["label"],
)
train_df.shape, test_df.shape

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, classification_report

baseline = Pipeline(
    [
        ("tfidf", TfidfVectorizer(max_features=20000, ngram_range=(1, 2))),
        ("clf", LogisticRegression(max_iter=1000)),
    ]
)

baseline.fit(train_df["text"], train_df["label"])
test_pred = baseline.predict(test_df["text"])

acc = accuracy_score(test_df["label"], test_pred)
print(f"Baseline TF-IDF + LogisticRegression accuracy: {acc:.3f}")
print(classification_report(test_df["label"], test_pred, target_names=[label_map[i] for i in sorted(label_map)]))

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch
import numpy as np
from sklearn.metrics import accuracy_score, classification_report

MODEL_NAME = "kdave/FineTuned_Finbert"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME)


def predict_batch(texts, batch_size=16, max_length=128):
    preds = []
    for start in range(0, len(texts), batch_size):
        batch = texts[start : start + batch_size]
        inputs = tokenizer(
            batch,
            padding=True,
            truncation=True,
            max_length=max_length,
            return_tensors="pt",
        )
        with torch.no_grad():
            outputs = model(**inputs)
            probs = torch.nn.functional.softmax(outputs.logits, dim=-1)
            preds.extend(torch.argmax(probs, dim=-1).tolist())
    return np.array(preds)


test_preds = predict_batch(test_df["text"].tolist())
acc = accuracy_score(test_df["label"], test_preds)
print(f"FinBERT accuracy: {acc:.3f}")
print(classification_report(test_df["label"], test_preds, target_names=[label_map[i] for i in sorted(label_map)]))